# 03 - Parsing the output and the mctal

**~30 minutes.**

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

In [ ]:
import matplotlib.pyplot as plt

from f4enix.input.MCNPinput import Input
from f4enix.output.mctal import Mctal
from f4enix.output.MCNPoutput import Output

## 1. Open the output

`Output` reads the whole file into memory when you build it. On this 1 MB `outp` it takes a
moment.

In [ ]:
out = Output(DATA / "iter_1d" / "outp")

print("code version:", out.get_code_version())
print("histories   :", f"{out.get_NPS():,}")

## 2. Warnings and fatal errors

`get_warnings()` returns a table of every warning MCNP printed, **grouped by type**, with how
many times each one occurred.

In [ ]:
warnings = out.get_warnings()
print("distinct warnings:", len(warnings))
warnings.head(10)

A quick way to filter:

In [ ]:
warnings[warnings.index.str.contains("tallies|cutoff|zero", case=False)]

Sort from most frequent to less frequent

In [ ]:
warnings.sort_values(ascending=False)

In [ ]:
# a run that died would list its reasons here
print("fatal errors:", out.get_fatal_errors())

## 3. Printed tables

Any table MCNP prints can be pulled out **by its table number** and comes back as a DataFrame.
Table 60 is the cell summary: material, densities, volumes, masses, importances.

In [ ]:
table60 = out.get_table(60)
print(table60.shape)
table60.head()

Because it is a DataFrame, you can immediately compute things MCNP does not print, like the
total mass of the model.

In [ ]:
mass = table60[" mass"].astype(float)
print(f"total mass: {mass.sum():.3e} g  ({mass.sum() / 1e6:.1f} tonnes)")

Table 126 is the particle activity per cell - the table you look at when you suspect a region
is not being sampled.

In [ ]:
table126 = out.get_table(126)
table126.head()

> Some tables are printed several times in one output (once per cycle, for instance).
> `get_table(126, instance_idx=-1)` gives you the **last** one, `instance_idx=0` the first.

## 4. The ten statistical checks

This is the part of the output everybody knows they should read and nobody does.
`get_stat_checks_table()` gives you one row per tally and one column per check.

In [ ]:
checks = out.get_stat_checks_table()
checks

Finding the tallies in trouble is now a one-liner: keep the rows where any column says `no`.

In [ ]:
failing = checks[(checks == "no").any(axis=1)]
print("tallies that missed a check:", list(failing.index))
failing

For one specific tally you get the full MCNP table - what was desired, what was observed, and
whether it passed.

In [ ]:
out.get_tally_stat_checks(74)

So tally 74 fails on the **figure of merit behaviour**.

The last column of the summary table, `Other TFC bins`, comes from a separate method. It tells
you whether *all* the bins of a tally passed, not just the total bin.

In [ ]:
summary_checks = out.get_statistical_checks_tfc_bins()
print(summary_checks)

Tally numbers alone are hard to discuss in a meeting. `assign_tally_description` replaces them
with the `FC` comment, taken from the `mctal`.

In [ ]:
mctal = Mctal(DATA / "iter_1d" / "mctal")

named = out.assign_tally_description(summary_checks, mctal.tallies)
for name, result in named.items():
    print(f"{result:8s} {name}")

## 5. Lost particles

`ITER_1D` is clean, so we switch to an output that is not: `test_lp_u.o`, with 677 lost
particles out of 10 010 histories.

In [ ]:
lp_out = Output(DATA / "extra" / "test_lp_u.o")
lp_inp = Input.from_input(DATA / "extra" / "test_lp_u.i")

print("lost particles :", lp_out.get_tot_lp())
print("histories      :", lp_out.get_NPS())
print("lost fraction  :", f"{lp_out.get_LPR():.2%}")

`get_lp_debug_df` gives you **where** each particle was lost: position, direction, cell,
surface. Passing the input as well adds the universe, which is what you need in a nested model.

In [ ]:
lp = lp_out.get_lp_debug_df(input_mcnp=lp_inp)
lp.head()

Grouping tells you straight away where the geometry problem is.

In [ ]:
lp.groupby(["Cell", "Surface"]).size().sort_values(ascending=False)

635 of the 677 losses happen at the same cell/surface pair. That is one bad boundary, not a
diffuse problem.

`print_lp_debug` writes the whole diagnosis to disk in three formats at once: an Excel summary,
a CSV with the flight directions, and a **VTK point cloud** you can drop into ParaView on top of
your geometry.

In [ ]:
lp_out.print_lp_debug(OUT, input_mcnp=lp_inp)

print([p.name for p in sorted(OUT.glob("LPdebug*"))])

## 6. The mctal: tally results

`Mctal` parses the tally file. The header carries the run metadata.

In [ ]:
print("code    :", mctal.header.kod, mctal.header.ver)
print("title   :", mctal.header.title)
print("histories:", f"{mctal.header.nps:,}")
print("tallies :", mctal.header.ntal)

`get_error_summary()` is the first thing to look at: the minimum and maximum relative error of
every tally, in one table.

In [ ]:
errors = mctal.get_error_summary()
errors

Tallies 4 and 14 show `NaN` and a maximum error of 1.0. That is the signature of **bins that
scored nothing**: they are the energy-binned flux tallies, and the highest energy bins are
simply empty.

In [ ]:
# every tally is a DataFrame in tallydata, keyed by its number
print(sorted(mctal.tallydata.keys()))

In [ ]:
mctal.tallydata[204].head()      # total neutron flux, one value per cell

## 7. Plotting a spectrum

Tally 4 is the energy-binned neutron flux, so its DataFrame has an extra `Energy` column.

In [ ]:
mctal.tallydata[4].head()

Each tally also carries `total` rows. They are useful, but they get in the way when plotting,
so we drop them first.

In [ ]:
print("with totals   :", mctal.tallydata[4].shape)
mctal.remove_totals(4)
print("without totals:", mctal.tallydata[4].shape)

flux = mctal.tallydata[4]

Now the plot is just "filter the rows of one cell, then draw them". Cell 47 is in the inboard
shield.

In [ ]:
spectrum = flux[flux["Cells"] == 47]

fig, ax = plt.subplots(figsize=(7, 4))
ax.step(spectrum["Energy"], spectrum["Value"], where="post")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy [MeV]")
ax.set_ylabel("Neutron flux [#/cm2], normalised by FM4")
ax.set_title("Neutron spectrum in cell 47")
ax.grid(alpha=0.3)